# CREAR LA BASE DE DATOS

In [1]:
from database.sql_functions import preparar_base_de_datos, obtener_lista_tablas, ejecutar_sql, obtener_esquema_como_dataframe
from database.datos_sinteticos import entrenar_y_poblar_todo

# 1. Crear tablas e insertar datos iniciales del .sql
preparar_base_de_datos("database/schema.sql")

total_datos = ejecutar_sql("SELECT COUNT(*) AS total_usuarios FROM users")

valor_total = total_datos.iloc[0, 0]
limite_datos = 50

if valor_total < limite_datos:
    print(f"Ampliando la cantidad de registros de {valor_total} hasta {limite_datos}")
    # 2. Generar y poblar datos sintéticos adicionales
    entrenar_y_poblar_todo(n_muestras_usuario=limite_datos)

# 3. Verificar que las tablas están listas
tablas = obtener_lista_tablas()
print(f'Tablas disponibles: {tablas}')

tablas_esquemas = obtener_esquema_como_dataframe(como_json=True)
display(tablas_esquemas)

# 4. Probar que hay más datos
ejecutar_sql("SELECT COUNT(*) AS total_usuarios FROM users")

Base de datos Generada, Correctamente a partir del archivo .sql
Tablas disponibles: ['users', 'categories', 'products', 'subscriptions', 'orders', 'order_items', 'event_logs']


,tabla,columnas
0,users,"{""user_id"": ""INTEGER (PK)"", ""full_name"": ""TEXT [NOT NULL]"", ""email"": ""TEXT [NOT NULL]"", ""country"": ""TEXT [NOT NULL]"", ""signup_date"": ""DATE [NOT NULL]"", ""referrer_id"": ""INTEGER"", ""profile_data"": ""JSON""}"
1,categories,"{""category_id"": ""INTEGER (PK)"", ""name"": ""TEXT [NOT NULL]""}"
2,products,"{""product_id"": ""INTEGER (PK)"", ""name"": ""TEXT [NOT NULL]"", ""category_id"": ""INTEGER [NOT NULL]"", ""price"": ""REAL [NOT NULL]"", ""cost"": ""REAL [NOT NULL]"", ""stock"": ""INTEGER [NOT NULL]""}"
3,subscriptions,"{""subscription_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER [NOT NULL]"", ""plan_tier"": ""TEXT"", ""start_date"": ""DATE [NOT NULL]"", ""end_date"": ""DATE"", ""monthly_fee"": ""REAL [NOT NULL]""}"
4,orders,"{""order_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER [NOT NULL]"", ""order_date"": ""DATETIME [NOT NULL]"", ""status"": ""TEXT"", ""payment_method"": ""TEXT [NOT NULL]""}"
5,order_items,"{""order_id"": ""INTEGER (PK) [NOT NULL]"", ""product_id"": ""INTEGER (PK) [NOT NULL]"", ""quantity"": ""INTEGER [NOT NULL]"", ""unit_price"": ""REAL [NOT NULL]"", ""discount"": ""REAL""}"
6,event_logs,"{""log_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER"", ""event_type"": ""TEXT [NOT NULL]"", ""event_timestamp"": ""DATETIME [NOT NULL]"", ""metadata"": ""JSON""}"


,total_usuarios
0,50


# PRODUCTOS QUE NO HAN SIDO COMPRADOS

In [2]:
query = """
SELECT products.*, categories.name AS category_name
    FROM products
INNER JOIN categories ON categories.category_id = products.category_id
    LEFT JOIN order_items ON products.product_id = order_items.product_id
WHERE order_items.product_id IS NULL;
"""

ejecutar_sql(query)

,product_id,name,category_id,price,cost,stock,category_name
0,49,Car Vacuum Cleaner Cordless,10,40.0,15.0,40,Automotive
1,50,Microfiber Cleaning Cloths,10,12.0,3.0,150,Automotive


In [3]:
query = """
SELECT products.*, categories.* 
    FROM products
RIGHT JOIN categories ON categories.category_id = products.category_id
    WHERE products.category_id IS NULL
"""

ejecutar_sql(query)

,product_id,name,category_id,price,cost,stock,category_id,name
0,None,None,None,None,None,None,11,Pet Supplies


# TOP 3 DE PRODUCTOS MAS COMPRADOS Y MENOS COMPARADOS

In [4]:
TOP_PRODUCTOS = 3
query = f"""
WITH ranked AS (
    SELECT 
        oi.product_id, 
        p.name, 
        COUNT(*) AS TOTAL,
        DENSE_RANK() OVER (ORDER BY COUNT(*) DESC) AS rnk
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY oi.product_id, p.name
)
SELECT product_id, name, TOTAL 
FROM ranked 
WHERE rnk <= {TOP_PRODUCTOS};
"""

ejecutar_sql(query)

,product_id,name,TOTAL
0,16,Smartwatch Fitness Track,118
1,12,Office Chair Ergonomic,106
2,4,Monitor 27 Inch,99


In [5]:
TOP_PRODUCTOS = 3
query = f"""
SELECT order_items.product_id, products.name, COUNT(*) AS TOTAL
    FROM order_items
JOIN products
ON products.product_id = order_items.product_id
GROUP BY order_items.product_id, products.name
ORDER BY TOTAL DESC
LIMIT {TOP_PRODUCTOS};
"""

ejecutar_sql(query)

,product_id,name,TOTAL
0,16,Smartwatch Fitness Track,118
1,12,Office Chair Ergonomic,106
2,4,Monitor 27 Inch,99


# TOP 3 DE PRODUCTOS MAS COMPRADOS POR CADA CATEGORIA

In [6]:
TOP_PRODUCTOS = 3

query = f"""
WITH ranked AS (
    SELECT 
        c.name AS category_name,
        oi.product_id, 
        p.name, 
        COUNT(oi.order_id) AS TOTAL,
        DENSE_RANK() OVER (PARTITION BY c.category_id ORDER BY COUNT(oi.order_id) DESC) AS rnk
    FROM categories c
    INNER JOIN products p ON c.category_id = p.category_id
    INNER JOIN order_items oi ON p.product_id = oi.product_id
    GROUP BY c.category_id, c.name, oi.product_id, p.name
)
SELECT category_name, product_id, name, TOTAL, rnk 
FROM ranked 
WHERE rnk <= {TOP_PRODUCTOS};
"""

ejecutar_sql(query)

,category_name,product_id,name,TOTAL,rnk
0,Electronics,4,Monitor 27 Inch,99,1
1,Electronics,2,Wireless Mouse,95,2
2,Electronics,5,Webcam 4K,90,3
3,Software,10,VPN Secure Pass,96,1
4,Software,6,Antivirus Annual License,90,2
5,Software,8,Photo Editor Pro Sub,87,3
6,Furniture,12,Office Chair Ergonomic,106,1
7,Furniture,13,Bookshelf Wooden,84,2
8,Furniture,15,Drawer Pedestal,84,2
9,Furniture,14,Meeting Table,78,3


# LISTAR LA ORDEN MAS RECIENTE PARA CADA USUARIO DE COLOMBIA

In [7]:
query = f"""
WITH user_latest_orders AS (
    SELECT 
        u.full_name,
        o.order_id,
        o.order_date,
        o.status,
        o.payment_method,
        ROW_NUMBER() OVER (PARTITION BY u.user_id ORDER BY o.order_date DESC) AS rnk
    FROM users u
    JOIN orders o ON u.user_id = o.user_id
    WHERE u.country = 'CO'
    --AND o.status = 'completed'
)
SELECT full_name, order_id, order_date, status, payment_method
FROM user_latest_orders
WHERE rnk = 1;
"""

ejecutar_sql(query)

,full_name,order_id,order_date,status,payment_method
0,Lucía Morales,1544,2026-12-21 20:02:24,cancelled,wire_transfer
1,Camila Vargas,1391,2026-12-05 16:24:26,pending,paypal
2,Andrés Ruiz,1851,2026-11-22 08:05:13,refunded,paypal
3,Manuel Bravo,1218,2026-12-26 15:14:34,completed,wire_transfer
4,Raquel Cortés,1767,2026-10-24 10:44:44,cancelled,credit_card
5,Jaime Pastor,1582,2026-12-24 16:40:13,completed,paypal


## TOP DE USUARIOS QUE MÁS HAN COMPRADO POR PAÍS, AÑO Y MES (EJ. JULIO DE 2026) CON LA COMPRA COMPLETADA, DETALLANDO TOTAL DE ÓRDENES, ARTÍCULOS (TOTAL, PROMEDIO, MÁXIMO, MÍNIMO), GASTO TOTAL, INGRESO NETO Y PRODUCTOS CONCATENADOS.

In [8]:
# -- SQLite / MySQL / MariaDB:           GROUP_CONCAT(p.name || ' (x' || oi.quantity || ')', ', ')
# -- PostgreSQL / SQL Server / BigQuery: STRING_AGG(p.name || ' (x' || oi.quantity || ')', ', ')
# -- Snowflake:                          LISTAGG(p.name || ' (x' || oi.quantity || ')', ', ')
# -- Oracle:                             LISTAGG(p.name || ' (x' || oi.quantity || ')', ', ') WITHIN GROUP (ORDER BY p.name)

query = """
WITH resumen_por_usuario_mensual AS (
    SELECT 
        u.full_name, 
        u.country,                                                          -- 👤 País del usuario
        strftime('%Y', o.order_date) AS year,                               -- 📅 Año de la compra
        strftime('%m', o.order_date) AS month,                              -- 📅 Mes de la compra
        COUNT(DISTINCT o.order_id) AS total_ordenes,                        -- 📦 Total de órdenes en el mes
        ROUND(AVG(oi.quantity), 2) AS promedio_articulos,
        MAX(oi.quantity) AS maximos_articulos,
        MIN(oi.quantity) AS minimos_articulos,
        SUM(oi.quantity) AS total_articulos,                                -- 🛒 Total de unidades compradas en el mes
        SUM((oi.unit_price - oi.discount) * oi.quantity) AS gasto_total,    -- 💰 Total gastado en el mes
        SUM((oi.unit_price - oi.discount - p.cost) * oi.quantity) AS ingreso_neto_total, -- 💰 Ingreso neto en el mes
        
        -- 🔗 Nombres de los productos y cantidades compradas en ese mes
        GROUP_CONCAT(p.name || ' (x' || oi.quantity || ')', ', ') AS productos_comprados,
        
        -- 🏆 Ranking por país, año y mes basado en el gasto total acumulado
        RANK() OVER (
            PARTITION BY u.country, strftime('%Y', o.order_date), strftime('%m', o.order_date) 
            ORDER BY SUM((oi.unit_price - oi.discount) * oi.quantity) DESC
        ) AS rnk
        
    FROM users u
        INNER JOIN orders o ON u.user_id = o.user_id
        INNER JOIN order_items oi ON o.order_id = oi.order_id
        INNER JOIN products p ON p.product_id = oi.product_id
    WHERE o.status = 'completed'
    GROUP BY u.user_id, u.full_name, u.country, strftime('%Y', o.order_date), strftime('%m', o.order_date)
)
SELECT 
    full_name, 
    country, 
    year,
    month,
    total_ordenes, 
    total_articulos, 
    maximos_articulos,
    minimos_articulos,
    promedio_articulos,
    gasto_total, 
    ingreso_neto_total,
    productos_comprados
FROM resumen_por_usuario_mensual
WHERE year = '2026' 
    AND month = '07' 
    AND rnk <= 1  -- 🔍 FILTRADO DIRECTAMENTE EN EL SELECT FINAL
ORDER BY month DESC, ingreso_neto_total DESC;
"""

ejecutar_sql(query)

,full_name,country,year,month,total_ordenes,total_articulos,maximos_articulos,minimos_articulos,promedio_articulos,gasto_total,ingreso_neto_total,productos_comprados
0,Martin Delgado,AR,2026,07,3,12,3,1,2.00,7235.00,2061.00,"Cargador Inalámbrico Fast (x2), Laptop Pro 15 (x2), Laptop Pro 15 (x1), Funda de Protección Grip (x1), USB-C Hub Multiport (x3), Laptop Pro 15 (x3)"
1,Francisco Reyes,MX,2026,07,1,4,2,2,2.00,3260.00,1500.00,"Webcam 4K (x2), Laptop Pro 15 (x2)"
2,Mariana Ortiz,ES,2026,07,1,5,2,1,1.67,2969.98,1214.98,"Funda de Protección Grip (x1), Laptop Pro 15 (x2), Lego City Building Set (x2)"
3,Emilio Santos,PE,2026,07,1,2,1,1,1.00,1630.00,750.00,"Webcam 4K (x1), Laptop Pro 15 (x1)"
4,Sofia Silva,CL,2026,07,2,8,2,1,1.60,719.98,591.98,"Cloud Storage 1TB/yr (x2), VPN Secure Pass (x2), Antivirus Annual License (x1), Photo Editor Pro Sub (x1), Electric Kettle Stainless (x2)"
5,Gonzalo Ibáñez,US,2026,07,1,4,2,2,2.00,400.00,196.00,"VPN Secure Pass (x2), Office Chair Ergonomic (x2)"
6,Lucía Morales,CO,2026,07,2,9,3,1,2.25,670.00,-528.00,"Antivirus Annual License (x2), Wireless Mouse (x1), Mechanical Keyboard (x3), Meeting Table (x3)"


# LISTE LOS REGISTROS DUPLICADOS DE event_logs

In [9]:
query = """
SELECT 
    user_id, 
    event_type, 
    event_timestamp, 
    metadata, 
    COUNT(*) AS cantidad
FROM event_logs
GROUP BY user_id, event_type, event_timestamp, metadata
HAVING COUNT(*) > 1;
"""

ejecutar_sql(query)

,user_id,event_type,event_timestamp,metadata,cantidad
0,1,search,2026-05-28 08:36:51,"{""query"": ""mouse""}",2
1,9,add_to_cart,2025-06-21 11:44:15,"{""product_id"": 10, ""device"": ""tablet""}",2
2,13,checkout_error,2026-05-14 03:31:46,"{""code"": 500, ""message"": ""Transaction failed""}",2
3,20,login,2025-11-10 07:35:35,"{""status"": ""success""}",2
4,47,login,2026-04-06 12:18:15,"{""status"": ""success""}",2


# CONSULTAS A BASE DE DATOS

In [10]:
obtener_esquema_como_dataframe(como_json=True)

,tabla,columnas
0,users,"{""user_id"": ""INTEGER (PK)"", ""full_name"": ""TEXT [NOT NULL]"", ""email"": ""TEXT [NOT NULL]"", ""country"": ""TEXT [NOT NULL]"", ""signup_date"": ""DATE [NOT NULL]"", ""referrer_id"": ""INTEGER"", ""profile_data"": ""JSON""}"
1,categories,"{""category_id"": ""INTEGER (PK)"", ""name"": ""TEXT [NOT NULL]""}"
2,products,"{""product_id"": ""INTEGER (PK)"", ""name"": ""TEXT [NOT NULL]"", ""category_id"": ""INTEGER [NOT NULL]"", ""price"": ""REAL [NOT NULL]"", ""cost"": ""REAL [NOT NULL]"", ""stock"": ""INTEGER [NOT NULL]""}"
3,subscriptions,"{""subscription_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER [NOT NULL]"", ""plan_tier"": ""TEXT"", ""start_date"": ""DATE [NOT NULL]"", ""end_date"": ""DATE"", ""monthly_fee"": ""REAL [NOT NULL]""}"
4,orders,"{""order_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER [NOT NULL]"", ""order_date"": ""DATETIME [NOT NULL]"", ""status"": ""TEXT"", ""payment_method"": ""TEXT [NOT NULL]""}"
5,order_items,"{""order_id"": ""INTEGER (PK) [NOT NULL]"", ""product_id"": ""INTEGER (PK) [NOT NULL]"", ""quantity"": ""INTEGER [NOT NULL]"", ""unit_price"": ""REAL [NOT NULL]"", ""discount"": ""REAL""}"
6,event_logs,"{""log_id"": ""INTEGER (PK)"", ""user_id"": ""INTEGER"", ""event_type"": ""TEXT [NOT NULL]"", ""event_timestamp"": ""DATETIME [NOT NULL]"", ""metadata"": ""JSON""}"


# APLANAR KEYS DE UN JSON ANIDADO Y CONVERTIRLOS EN UNA TABLA

In [11]:
CONSULTA = f"""
SELECT 
    u.user_id,
    u.full_name,
    jt.fullkey AS ruta_json,
    jt.key AS propiedad,
    jt.value AS valor,
    jt.type AS tipo_dato
FROM users u,
json_tree(u.profile_data) AS jt
WHERE jt.type NOT IN ('object', 'array');
"""
# Ejecución simple llamando a la función
df_resultado = ejecutar_sql(CONSULTA)

# Mostrar el DataFrame resultante
df_resultado

,user_id,full_name,ruta_json,propiedad,valor,tipo_dato
0,1,Carlos Mendoza,$.age,age,34,integer
1,1,Carlos Mendoza,$.preferences.theme,theme,dark,text
2,1,Carlos Mendoza,$.preferences.newsletter,newsletter,1,true
3,1,Carlos Mendoza,$.tags[0],0,vip,text
4,1,Carlos Mendoza,$.tags[1],1,tech,text
5,2,Ana Gómez,$.age,age,28,integer
6,2,Ana Gómez,$.preferences.theme,theme,light,text
7,2,Ana Gómez,$.preferences.newsletter,newsletter,0,false
8,2,Ana Gómez,$.tags[0],0,casual,text
9,3,Luis Fernández,$.age,age,42,integer
